### Gold Layer Data Quality Check Notebook

Validates the Gold layer star schema for accuracy, completeness, and referential integrity before it is used for reporting.

**Source:** Gold Lakehouse (PROCUREMENT schema) &nbsp;&nbsp;|&nbsp;&nbsp; **Target:** Validation output only — no data is written

**Input:** gold_fact_procurement, gold_dim_vendor, gold_dim_material, gold_dim_company, gold_dim_plant, gold_dim_cost_center

**Transformations:** duplicate fact record detection · date dimension completeness check · referential integrity check between fact and dimension keys

**Output:** data quality validation report (printed results)

**Import PySpark functions and types**

In [1]:
from pyspark.sql.functions import*
from pyspark.sql.types import*

StatementMeta(, 8d2ff183-a080-4e65-9e71-7d4e22bac7ae, 3, Finished, Available, Finished, False)

**Read Gold layer fact and dimension tables**

In [2]:
# ==========================================================
# GOLD LAYER VALIDATION
# ==========================================================

# Read Gold Tables
df_Fact = spark.table("PROCUREMENT.gold_fact_procurement")
df_Vendor = spark.table("PROCUREMENT.gold_dim_vendor")
df_Material = spark.table("PROCUREMENT.gold_dim_material")
df_Company = spark.table("PROCUREMENT.gold_dim_company")
df_Plant = spark.table("PROCUREMENT.gold_dim_plant")
df_Cost_Center = spark.table("PROCUREMENT.gold_dim_cost_center")
df_Purchase_Group = spark.table("PROCUREMENT.gold_dim_purchase_group")
df_Currency = spark.table("PROCUREMENT.gold_dim_currency")
df_Date = spark.table("PROCUREMENT.gold_dim_date")

print("="*80)
print("GOLD LAYER DATA VALIDATION")
print("="*80)

StatementMeta(, 8d2ff183-a080-4e65-9e71-7d4e22bac7ae, 4, Finished, Available, Finished, False)

GOLD LAYER DATA VALIDATION


**Preview the Gold fact table**

In [3]:
display(df_Fact)

StatementMeta(, 8d2ff183-a080-4e65-9e71-7d4e22bac7ae, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e1f6b51d-1f3e-4aba-9947-23d6f0be55ad)

**Check for duplicate fact records and validate date dimension completeness**

In [4]:
# ==========================================================
# 1. Duplicate Fact Records
# ==========================================================

duplicate_fact = (df_Fact.groupBy("PO_Number", "PO_Line_Item")
.count().filter("count > 1").count())

print(f"Duplicate Fact Records : {duplicate_fact}")

# ==========================================================
# 2. Date Dimension Validation
# ==========================================================

total_dates = df_Date.count()
unique_dates = df_Date.select("Date_Key").distinct().count()

print(f"Date Dimension Total Rows            : {total_dates}")
print(f"Date Dimension Unique Date Keys      : {unique_dates}")

# ==========================================================
# 3. Missing Vendor
# ==========================================================

missing_vendor = df_Fact.join(
    df_Vendor.select("Vendor_ID"),
    "Vendor_ID",
    "left_anti"
).count()

print(f"Missing Vendor Records               : {missing_vendor}")

# ==========================================================
# 4. Missing Material
# ==========================================================

missing_material = df_Fact.join(
    df_Material.select("Material_ID"),
    "Material_ID",
    "left_anti"
).count()

print(f"Missing Material Records             : {missing_material}")

# ==========================================================
# 5. Missing Company
# ==========================================================

missing_company = df_Fact.join(
    df_Company.select("Company_Code"),
    "Company_Code",
    "left_anti"
).count()

print(f"Missing Company Records              : {missing_company}")

# ==========================================================
# 6. Missing Plant
# ==========================================================

missing_plant = df_Fact.join(
    df_Plant.select("Plant_ID"),
    "Plant_ID",
    "left_anti"
).count()

print(f"Missing Plant Records                : {missing_plant}")

# ==========================================================
# 7. Missing Cost Center
# ==========================================================

missing_cc = df_Fact.join(
    df_Cost_Center.select("Cost_Center_ID"),
    "Cost_Center_ID",
    "left_anti"
).count()

print(f"Missing Cost Center Records          : {missing_cc}")

# ==========================================================
# 8. Missing Purchasing Group
# ==========================================================

missing_pg = df_Fact.join(
    df_Purchase_Group.select("Purchasing_Group_ID"),
    "Purchasing_Group_ID",
    "left_anti"
).count()

print(f"Missing Purchasing Group Records     : {missing_pg}")

# ==========================================================
# 9. Missing Currency
# ==========================================================

missing_currency = df_Fact.join(
    df_Currency.select("Currency_Code"),
    df_Fact.PO_Currency_Code == df_Currency.Currency_Code,
    "left_anti"
).count()

print(f"Missing Currency Records             : {missing_currency}")

# ==========================================================
# 10. Missing Date
# ==========================================================

missing_date = df_Fact.join(
    df_Date.select("Date_Key"),
    df_Fact.PO_Date_Key == df_Date.Date_Key,
    "left_anti"
).count()

print(f"Missing Date Records                 : {missing_date}")

print("="*80)

if (duplicate_fact == 0 and
    total_dates == unique_dates and
    missing_vendor == 0 and
    missing_material == 0 and
    missing_company == 0 and
    missing_plant == 0 and
    missing_cc == 0 and
    missing_pg == 0 and
    missing_currency == 0 and
    missing_date == 0):

    print("GOLD LAYER VALIDATION PASSED")
    print("Ready for Power BI")

else:

    print("Validation completed. Review the values above before proceeding.")

print("="*80)

StatementMeta(, 8d2ff183-a080-4e65-9e71-7d4e22bac7ae, 6, Finished, Available, Finished, False)

Duplicate Fact Records : 0
Date Dimension Total Rows            : 1461
Date Dimension Unique Date Keys      : 1461
Missing Vendor Records               : 0
Missing Material Records             : 0
Missing Company Records              : 0
Missing Plant Records                : 0
Missing Cost Center Records          : 251
Missing Purchasing Group Records     : 0
Missing Currency Records             : 2
Missing Date Records                 : 0
Validation completed. Review the values above before proceeding.


**Check for fact records with no matching Plant or Purchasing Group dimension**

In [5]:
# Find Fact records with no matching Purchasing Group

df_missing_pg = df_Fact.join(
    df_Plant.select("Plant_ID"),
    on="Plant_ID",
    how="left_anti"
)

print("Missing Purchasing Group Records :", df_missing_pg.count())

df_missing_pg.select(
    "PO_Number",
    "PO_Line_Item",
    "Plant_Id"
).show(truncate=False)

StatementMeta(, 8d2ff183-a080-4e65-9e71-7d4e22bac7ae, 7, Finished, Available, Finished, False)

Missing Purchasing Group Records : 0
+---------+------------+--------+
|PO_Number|PO_Line_Item|Plant_Id|
+---------+------------+--------+
+---------+------------+--------+



**Preview the final validated Gold fact table**

In [6]:
display(df_Fact)

StatementMeta(, 8d2ff183-a080-4e65-9e71-7d4e22bac7ae, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d7cde35a-35cb-4a6b-a3fa-121b63ee8d72)